# Binary Gesture Demo

This notebook shows the first model pipeline for the supervisor demo.

Demo story: the user draws **bird**, the bird sound starts. Then the user draws **fish**, and the river sound is added on top. In the dataset, the user-facing fish gesture is stored as the model label `river`.

## 1. Imports

The model here is intentionally simple. For tomorrow, the goal is to show that the data can separate two gestures and trigger two sound layers.

In [ ]:
import json
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score


## 2. Demo Settings

Change `REQUESTED_GESTURES` if you want to test another pair. User-facing names such as `fish`, `wave`, and `cloud` are mapped to the dataset labels.

In [ ]:
DATA_PATH = Path("data/training_samples_full.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/training_samples_full.json")


REQUESTED_GESTURES = ("bird", "fish")
TEST_SIZE = 0.25
SEED = 1

USER_TO_MODEL_LABEL = {
    "leaf": "leaf",
    "tree": "tree",
    "bird": "bird",
    "wave": "ocean",
    "ocean": "ocean",
    "fish": "river",
    "river": "river",
    "cloud": "rain",
    "rain": "rain",
}

CHANNELS = ("ax", "ay", "az", "gx", "gy", "gz")
TARGET_FRAMES = 151


## 3. Load the Cleaned Data

This uses the full JSON file we created from the approved valid ranges. It does not train from every raw recording.

In [ ]:
with DATA_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

all_samples = data["samples"]

label_counts = {}
for sample in all_samples:
    label_counts[sample["label"]] = label_counts.get(sample["label"], 0) + 1

print(f"Total approved samples: {len(all_samples)}")
print(label_counts)


## 4. Filter Bird and Fish

`fish` becomes `river` because the fish gesture is linked to the river sound in the current dataset.

In [ ]:
def normalize_label(label):
    key = label.strip().lower()
    if key not in USER_TO_MODEL_LABEL:
        known = ", ".join(sorted(USER_TO_MODEL_LABEL))
        raise ValueError(f"Unknown gesture '{label}'. Known gestures: {known}")
    return USER_TO_MODEL_LABEL[key]


model_labels = tuple(normalize_label(gesture) for gesture in REQUESTED_GESTURES)
samples = [sample for sample in all_samples if sample["label"] in set(model_labels)]

selected_counts = {}
for sample in samples:
    selected_counts[sample["label"]] = selected_counts.get(sample["label"], 0) + 1

print(f"Requested gestures: {REQUESTED_GESTURES}")
print(f"Model labels: {model_labels}")
print(f"Selected samples: {len(samples)}")
print(selected_counts)


## 5. Convert IMU Frames to Features

Each recording is about 3 seconds at 50 Hz. We keep a fixed 151-frame window and use the six watch IMU channels: accelerometer and gyroscope x/y/z.

In [ ]:
def fixed_length_sequence(frames):
    values = np.array([[frame[channel] for channel in CHANNELS] for frame in frames], dtype=float)

    if len(values) < TARGET_FRAMES:
        pad = np.repeat(values[-1:], TARGET_FRAMES - len(values), axis=0)
        values = np.vstack([values, pad])

    return values[:TARGET_FRAMES]


def features_for_sample(sample):
    sequence = fixed_length_sequence(sample["frames"])

    accel_mag = np.linalg.norm(sequence[:, :3], axis=1, keepdims=True)
    gyro_mag = np.linalg.norm(sequence[:, 3:], axis=1, keepdims=True)
    delta = np.diff(sequence, axis=0, prepend=sequence[:1])

    sequence_features = np.hstack([sequence, accel_mag, gyro_mag, delta]).reshape(-1)

    summary_parts = []
    for block in (sequence, delta):
        summary_parts.extend([
            block.mean(axis=0),
            block.std(axis=0),
            block.min(axis=0),
            block.max(axis=0),
            np.sqrt((block * block).mean(axis=0)),
        ])

    return np.concatenate([sequence_features, *summary_parts])


x = np.array([features_for_sample(sample) for sample in samples])
y = np.array([sample["label"] for sample in samples])

print(f"Feature matrix: {x.shape}")
print(f"Labels: {sorted(set(y))}")


## 6. Train the First Classifier

Random Forest is good enough for the demo because it trains quickly and works well on this small feature set.

In [ ]:
model = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    class_weight="balanced",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(model, x, y, cv=cv)

split = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_index, test_index = next(split.split(x, y))

model.fit(x[train_index], y[train_index])
predictions = model.predict(x[test_index])

accuracy = accuracy_score(y[test_index], predictions)
matrix = confusion_matrix(y[test_index], predictions, labels=list(model_labels))

print(f"5-fold CV accuracy: {cv_scores.mean():.3f}")
print(f"Held-out accuracy: {accuracy:.3f}")
print(list(model_labels))
print(matrix)


## 7. Classification Report

This gives precision and recall for each gesture. For the demo, the confusion matrix is usually the clearest thing to show.

In [ ]:
print(classification_report(
    y[test_index],
    predictions,
    labels=list(model_labels),
    zero_division=0,
))


## 8. Audio Layer Trigger Story

The web/app behavior for the demo can be explained with this simple mapping. The model classifies the gesture, then the audio engine starts or keeps the matching layer.

In [ ]:
SOUND_LAYER = {
    "bird": "bird chirps",
    "river": "river sound",
}

demo_sequence = ["bird", "river"]
active_layers = []

for predicted_label in demo_sequence:
    layer = SOUND_LAYER[predicted_label]
    if layer not in active_layers:
        active_layers.append(layer)
    print(f"Detected {predicted_label} -> playing layers: {active_layers}")
